In [ ]:

from features import CREDIT_CARD_COLS,INSTALLMENTS_COLS


# MERGE AU NIVEAU CRÉDIT (SK_ID_PREV)

# On merge les deux tables agrégées au niveau crédit
credit_prev_merged = credit_card_client.merge(
    installments_agg,
    on='SK_ID_PREV',
    how='outer',  # On garde tous les crédits (certains n'ont pas de carte, d'autres pas d'échéancier)
    indicator=True  # Pour vérifier la qualité du merge
)

# Vérifier la qualité du merge
print(credit_prev_merged['_merge'].value_counts())
# both : crédit avec carte ET échéancier
# left_only : crédit avec carte mais SANS échéancier
# right_only : crédit avec échéancier mais SANS carte

credit_prev_merged.drop('_merge', axis=1, inplace=True)


# AGRÉGATION FINALE PAR CLIENT (SK_ID_CURR)

# On agrège au niveau client
credit_client_final = credit_prev_merged.drop('SK_ID_PREV', axis=1).groupby('SK_ID_CURR').agg(['mean', 'max', 'min', 'sum'])
credit_client_final.columns = ['CREDIT_' + '_'.join(col).upper() for col in credit_client_final.columns]

# Ajouter des features de comptage
credit_client_final['CREDIT_NUM_PREV'] = credit_prev_merged.groupby('SK_ID_CURR')['SK_ID_PREV'].nunique()
credit_client_final['CREDIT_NUM_CC'] = credit_prev_merged.groupby('SK_ID_CURR')['SK_ID_PREV'].apply(
    lambda x: credit_prev_merged.loc[x.index, 'CC_COUNT_RECORDS'].notna().sum()
)
credit_client_final['CREDIT_NUM_INSTAL'] = credit_prev_merged.groupby('SK_ID_CURR')['INSTAL_COUNT'].notna().sum()

credit_client_final.reset_index(inplace=True)

print(f"\n✅ Shape finale par client : {credit_client_final.shape}")

NameError: name 'credit_card_prev' is not defined